# Reinforcement Learning for Pick and Place Tasks

**Objective:** In this notebook, we will learn the fundamental pipeline of training an RL agent for a robotic manipulation task:
1.  **Environment Setup:** Initializing and wrapping a custom robotic environment (`UR5RobotiqEnv`) for the pick-and-place task.
2.  **Agent Training:** Training Deep Reinforcement Learning algorithms (such as SAC, PPO, or A2C) using Stable Baselines3, including managing custom training loops and saving checkpoints.
3.  **Evaluation and Visualization:** Automatically recording and organizing video rollouts during both the training process and the final evaluation to visually monitor the robot's learning progress.

In [ ]:
# Installation prerrequisites
!pip install stable-baselines3
!pip install pybullet
!pip install numpy
!pip install matplotlib
!pip install imageio-ffmpeg


!git clone https://github.com/Mobile-Robots-Group-UC3M/ReinforcementLearningPickPlace.git
%cd ReinforcementLearningPickPlace

In [1]:
# Importing necessary libraries
from ur5_env_test import  UR5RobotiqEnv
from stable_baselines3 import PPO, SAC,A2C
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback
import time
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Video recording imports
import imageio_ffmpeg
from IPython.display import HTML
from base64 import b64encode

pybullet build time: Jan 29 2025 23:17:20


## 1. Training Reinforcement Learning Algorithms

Robots learning manipulation tasks usually rely on **Reinforcement Learning (RL)** framed within a simulated environment. Here, the agent learns how to solve the task by interacting with the world through continuous trial and error.
* **Algorithms:** We utilize state-of-the-art Deep RL algorithms such as **SAC (Soft Actor-Critic)**, **PPO (Proximal Policy Optimization)**, or **A2C (Advantage Actor-Critic)** to optimize the robot's policy.
* **Environment:** A custom physics-based simulation (`UR5RobotiqEnv`) where a UR5 robot arm equipped with a gripper must learn to grasp an object and move it to a specific target location.
* **State, Action & Reward:** At each step, the agent observes the current *state* (e.g., joint angles, object coordinates), outputs an *action* (e.g., motor velocities, gripper control), and receives a *reward* signal (e.g., positive feedback for successful placement, penalties for dropping the object or taking too long).

In [ ]:
def train_algo(record_video=True):
    # Create directories to store the logs, videos, and models if they don't exist
    log_dir = "./logs"
    train_video_dir = "videos/train"
    model_dir = "./models"
    
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(train_video_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    # Wrap the environment with the Monitor to record the results
    env = UR5RobotiqEnv()  # Instantiate the custom driving environment
    env = Monitor(env, log_dir)  # Monitor the environment
    
    # Predefine the algorithm to use (PPO, SAC, or A2C)
    algo_name = "SAC"  
    if algo_name == "PPO":
        model = PPO("MlpPolicy", env, verbose=1) # CHANGE HYPERPARAMETERS HERE
    elif algo_name == "SAC":
        model = SAC("MlpPolicy", env, verbose=1) # CHANGE HYPERPARAMETERS HERE
    elif algo_name == "A2C":
        model = A2C("MlpPolicy", env, verbose=1) # CHANGE HYPERPARAMETERS HERE
    else:
        raise ValueError("Invalid algorithm name. Please choose 'PPO', 'SAC', or 'A2C'.")

    # Configure the steps
    total_timesteps = 10000
    save_freq = 1000
    iterations = total_timesteps // save_freq  # This will result in 10 iterations

    print(f"Starting training: {iterations} iterations of {save_freq} steps.")

    # Manual loop to save model and video every 'save_freq' steps
    for i in range(1, iterations + 1):
        current_steps = i * save_freq
        
        # 1. Start video recorder for this iteration
        vid = None
        if record_video:
            env.unwrapped.cam_width, env.unwrapped.cam_height = 1270, 720
            video_path = os.path.join(train_video_dir, f'train_video_{current_steps}_steps.mp4')
            
            vid = imageio_ffmpeg.write_frames(video_path, (env.unwrapped.cam_width, env.unwrapped.cam_height), fps=30)
            vid.send(None) # seed the video writer
            env.unwrapped.video_writer = vid

        # 2. Train the model (reset_num_timesteps=False is CRUCIAL to continue from where it left off)
        model.learn(total_timesteps=save_freq, reset_num_timesteps=False, progress_bar=True)

        # 3. Save the model (Replaces the need for CheckpointCallback)
        model_path = os.path.join(model_dir, f"ur_robot_{algo_name.lower()}_{current_steps}_steps")
        model.save(model_path)
        
        # 4. Close the video for this iteration
        if record_video:
            env.unwrapped.video_writer = None
            vid.close()
            print(f"Training video saved on: {video_path}")

    # Save the final general model
    model.save(f"ur_robot_{algo_name.lower()}")
    print("Training completed!")

def test_algo():
    # 1. Create directory for evaluation videos
    eval_video_dir = "videos/evaluation"
    os.makedirs(eval_video_dir, exist_ok=True)

    env = UR5RobotiqEnv()
    
    # 2. Configure resolution and video writer
    env.cam_width, env.cam_height = 1270, 720 
    
    video_path = os.path.join(eval_video_dir, 'eval_video.mp4')
    vid = imageio_ffmpeg.write_frames(video_path, (env.cam_width, env.cam_height), fps=30)
    vid.send(None)
    env.video_writer = vid 

    # 3. Load the model
    algo_name = "SAC"
    model_path = f"./models/ur_robot_{algo_name.lower()}_7000_steps" # CHANGE THIS TO THE DESIRED MODEL PATH
    
    
    model = SAC.load(model_path, env=env)

    print("Starting video recording for EXACTLY ONE try...")
    obs, info = env.reset()
    done = False
    
    # 4. Evaluation loop
    while not done: 
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    print("Finished 1 try! Saving video...")
    
    # 5. Safe closing
    env.video_writer = None
    vid.close()
    env.close()
    print(f"Evaluation video saved to: {video_path}")

## 2. Reward function

Function to plot the results from the policy learned

In [3]:
# Function to plot the training reward data
def plot_reward_data():
    # Set the directory for storing Monitor log files
    log_dir = "./logs"
    files = {
        "PPO": "monitor_ppo.csv",
        "A2C": "monitor_a2c.csv",
        "SAC": "monitor_sac.csv"
    }

    # Create a plot for the rewards over time
    plt.figure(figsize=(12, 6))

    # Iterate through the files for each algorithm
    for label, file_name in files.items():
        monitor_file = os.path.join(log_dir, file_name)
        
        # Read the Monitor log data (skip the first row of comments)
        data = pd.read_csv(monitor_file, skiprows=1)
        
        # Calculate the cumulative timestep
        data['timestep'] = np.cumsum(data['l'])  # Accumulate the timestep, as global timestep

        data['timestep'] -= data['timestep'].iloc[0]  
        
        # Apply smoothing to the reward data
        smoothed_rewards = smooth(data['r'])
        
        # Plot the smoothed rewards against the timestep
        plt.plot(
            data['timestep'][:len(smoothed_rewards)], 
            smoothed_rewards, 
            label=label
        )
    
    # Add labels and title to the plot
    plt.xlabel("Step")
    plt.ylabel("Reward")
    plt.title("Training Reward")
    
    # Add legend to the plot
    plt.legend()
    plt.grid()  # Display grid lines
    plt.show()  # Show the plot
    
def smooth(data, window_size=10):
    """Apply weighted moving average smoothing to the data"""
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

## 3. Main loop

Here you can choose if you want to train or to test

In [7]:
# Main entry point for the script
if __name__ == '__main__':
    #train model
    #train_algo()  # Uncomment this line to train the model
    #test model
    test_algo()  # Call the test function to evaluate the trained model
    #plot_reward_data()  # Plot the training reward data

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1270, 720) to (1280, 720) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
ee_linkWrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Starting video recording for EXACTLY ONE try...
Finished 1 try! Saving video...
Video saved as test_video.mp4


## 4. Example of video from a policy learned

Here we present a example of what you should have after each training. All the videos are located in the folder [videos](/videos/)

In [8]:
mp4 = open('videos/evaluation/test_video.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML('<video width=480 controls><source src="%s" type="video/mp4"></video>' % data_url)